HW3: Search Algorithms
ใช้ romania_map, GraphProblem และ Node จากโน้ตบุ๊กเดิม

In [10]:
from search import Node, GraphProblem, UndirectedGraph
from collections import deque
from heapq import heappush, heappop
from itertools import count
from math import inf

# Romania map
romania_map = UndirectedGraph(dict(
    Arad=dict(Zerind=75, Sibiu=140, Timisoara=118),
    Bucharest=dict(Urziceni=85, Pitesti=101, Giurgiu=90, Fagaras=211),
    Craiova=dict(Drobeta=120, Rimnicu=146, Pitesti=138),
    Drobeta=dict(Mehadia=75),
    Eforie=dict(Hirsova=86),
    Fagaras=dict(Sibiu=99),
    Hirsova=dict(Urziceni=98),
    Iasi=dict(Vaslui=92, Neamt=87),
    Lugoj=dict(Timisoara=111, Mehadia=70),
    Oradea=dict(Zerind=71, Sibiu=151),
    Pitesti=dict(Rimnicu=97),
    Rimnicu=dict(Sibiu=80),
    Urziceni=dict(Vaslui=142)
))

# ใช้สำหรับ heuristic h(n) ของ Greedy, A* และ RBFS
romania_map.locations = dict(
    Arad=(91, 492), Bucharest=(400, 327), Craiova=(253, 288),
    Drobeta=(165, 299), Eforie=(562, 293), Fagaras=(305, 449),
    Giurgiu=(375, 270), Hirsova=(534, 350), Iasi=(473, 506),
    Lugoj=(165, 379), Mehadia=(168, 339), Neamt=(406, 537),
    Oradea=(131, 571), Pitesti=(320, 368), Rimnicu=(233, 410),
    Sibiu=(207, 457), Timisoara=(94, 410), Urziceni=(456, 350),
    Vaslui=(509, 444), Zerind=(108, 531)
)

START = "Arad"
GOAL = "Vaslui"

problem = GraphProblem(START, GOAL, romania_map)

Search Code

In [12]:
def in_path(node, state):
    return any(n.state == state for n in node.path())


# 1. Breadth-First Tree Search
def hw3_bfs_tree(problem):
    frontier = deque([Node(problem.initial)])

    while frontier:
        node = frontier.popleft()

        if problem.goal_test(node.state):
            return node

        for child in node.expand(problem):
            # กันการวนกลับในเส้นทางเดิม
            if not in_path(node, child.state):
                frontier.append(child)


# 2. Depth-First Tree Search
def hw3_dfs_tree(problem):
    frontier = [Node(problem.initial)]

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node

        children = node.expand(problem)
        for child in reversed(children):
            if not in_path(node, child.state):
                frontier.append(child)


# 3. Breadth-First Graph Search
def hw3_bfs_graph(problem):
    frontier = deque([Node(problem.initial)])
    visited = {problem.initial}

    while frontier:
        node = frontier.popleft()

        if problem.goal_test(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in visited:
                visited.add(child.state)
                frontier.append(child)


# 4. Depth-First Graph Search
def hw3_dfs_graph(problem):
    frontier = [Node(problem.initial)]
    visited = {problem.initial}

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node

        children = node.expand(problem)
        for child in reversed(children):
            if child.state not in visited:
                visited.add(child.state)
                frontier.append(child)


# 5. Generic Best-First Graph Search
# กำหนด f(node) ตอนเรียกใช้
def hw3_best_first_graph(problem, f):
    frontier = []
    order = count()

    root = Node(problem.initial)
    heappush(frontier, (f(root), next(order), root))

    best_cost = {root.state: root.path_cost}

    while frontier:
        _, _, node = heappop(frontier)

        if node.path_cost != best_cost.get(node.state):
            continue

        if problem.goal_test(node.state):
            return node

        for child in node.expand(problem):
            if child.path_cost < best_cost.get(child.state, inf):
                best_cost[child.state] = child.path_cost
                heappush(frontier, (f(child), next(order), child))


# 6. Uniform-Cost Search: f(n) = g(n)
def hw3_ucs(problem):
    return hw3_best_first_graph(problem, lambda node: node.path_cost)


# 7. Depth-Limited Search
def _hw3_dls(problem, limit):
    def visit(node, ancestors):
        if problem.goal_test(node.state):
            return node, False

        if node.depth == limit:
            return None, True

        cutoff = False

        for child in node.expand(problem):
            if child.state not in ancestors:
                result, was_cutoff = visit(
                    child,
                    ancestors | {child.state}
                )

                if result is not None:
                    return result, False

                cutoff = cutoff or was_cutoff

        return None, cutoff

    return visit(Node(problem.initial), {problem.initial})


def hw3_dls(problem, limit=5):
    node, _ = _hw3_dls(problem, limit)
    return node


# 8. Iterative Deepening Search
def hw3_ids(problem, max_depth=20):
    for limit in range(max_depth + 1):
        node, _ = _hw3_dls(problem, limit)

        if node is not None:
            return node


# 9. Greedy Best-First Search: f(n) = h(n)
def hw3_greedy(problem):
    return hw3_best_first_graph(
        problem,
        lambda node: problem.h(node)
    )


# 10. A* Search: f(n) = g(n) + h(n)
def hw3_astar(problem):
    return hw3_best_first_graph(
        problem,
        lambda node: node.path_cost + problem.h(node)
    )


# 11. Recursive Best-First Search
def hw3_rbfs(problem):
    def visit(node, ancestors, node_f, f_limit):
        if problem.goal_test(node.state):
            return node, node_f

        successors = []

        for child in node.expand(problem):
            if child.state not in ancestors:
                child_f = max(
                    child.path_cost + problem.h(child),
                    node_f
                )
                successors.append([child, child_f])

        if not successors:
            return None, inf

        while True:
            successors.sort(key=lambda x: x[1])

            best, best_f = successors[0]

            if best_f > f_limit:
                return None, best_f

            alternative = successors[1][1] if len(successors) > 1 else inf

            result, new_f = visit(
                best,
                ancestors | {best.state},
                best_f,
                min(f_limit, alternative)
            )

            successors[0][1] = new_f

            if result is not None:
                return result, new_f

    root = Node(problem.initial)
    result, _ = visit(root, {root.state}, problem.h(root), inf)
    return result

ผลลัพธ์

In [13]:
def show_result(name, node):
    if node is None:
        print(f"{name}: ไม่พบเส้นทาง")
        return

    route = [n.state for n in node.path()]

    print(f"\n{name}")
    print("Route :", " → ".join(route))
    print("Cost  :", node.path_cost)


show_result("1. Breadth-First Tree Search", hw3_bfs_tree(problem))
show_result("2. Depth-First Tree Search", hw3_dfs_tree(problem))
show_result("3. Breadth-First Graph Search", hw3_bfs_graph(problem))
show_result("4. Depth-First Graph Search", hw3_dfs_graph(problem))
show_result("5. Best-First Graph Search",hw3_best_first_graph(problem,lambda node: node.path_cost + problem.h(node)))
show_result("6. Uniform-Cost Search", hw3_ucs(problem))
show_result("7. Depth-Limited Search, limit=5", hw3_dls(problem, limit=5))
show_result("8. Iterative Deepening Search", hw3_ids(problem))
show_result("9. Greedy Best-First Search", hw3_greedy(problem))
show_result("10. A* Search", hw3_astar(problem))
show_result("11. Recursive Best-First Search", hw3_rbfs(problem))


1. Breadth-First Tree Search
Route : Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
Cost  : 677

2. Depth-First Tree Search
Route : Arad → Zerind → Oradea → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
Cost  : 834

3. Breadth-First Graph Search
Route : Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
Cost  : 677

4. Depth-First Graph Search
Route : Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
Cost  : 677

5. Best-First Graph Search
Route : Arad → Sibiu → Rimnicu → Pitesti → Bucharest → Urziceni → Vaslui
Cost  : 645

6. Uniform-Cost Search
Route : Arad → Sibiu → Rimnicu → Pitesti → Bucharest → Urziceni → Vaslui
Cost  : 645

7. Depth-Limited Search, limit=5
Route : Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
Cost  : 677

8. Iterative Deepening Search
Route : Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
Cost  : 677

9. Greedy Best-First Search
Route : Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
Cost  : 677

10. A* Search
Route : 